# Section 4 - Syntax Summary

## A/B testing and causality

Everything here runs on its own - the data is built
in the notebook rather than loaded from a file, so you can change a number and
re-run.

**Chapters 12.1 to 12.3 of the textbook.**


In [ ]:
# Run this cell to set up the notebook, but please don't change it.
%pip install -q urllib3<2.0 otter-grader datascience ipywidgets
try:
    import pyodide_http
    pyodide_http.patch_all()
except ImportError:
    pass

import numpy as np
from datascience import *

import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
import warnings
warnings.simplefilter('ignore', FutureWarning)

---

## Contents

1. [What an A/B test is for](#1)
2. [Step 1 - explore the two groups](#2)
3. [Step 2 - the observed difference](#3)
4. [Step 3 - simulating under the null by shuffling](#4)
5. [Step 4 - the p-value](#5)
6. [One-sided or two-sided?](#6)
7. [Why shuffling is not enough: confounding](#7)
8. [Randomised controlled trials](#8)
9. [Quick reference](#9)


---

<a id='1'></a>
## 1. What an A/B test is for

You have one sample split into two groups, and a numerical measurement on each
individual. The two groups have different averages. **Could that difference be
chance alone?**

That is the only question an A/B test answers.

| | |
|---|---|
| **Null** | The two groups come from the same underlying distribution. The difference is chance. |
| **Alternative** | They do not. Something systematic separates them. |

The test is also called a **permutation test**, because of how the simulation
works - you permute the labels.

We will use a study of birth weight and maternal smoking. The real data has
1,174 births; this is a smaller version with the same structure.


In [ ]:
# A stand-in for baby.csv, built so the two groups differ the way the real study does
np.random.seed(4)
n_smoke, n_non = 459, 715
births = Table().with_columns(
    'Maternal Smoker', np.append(np.repeat(True, n_smoke), np.repeat(False, n_non)),
    'Birth Weight',    np.append(np.round(np.random.normal(113.8, 18.3, n_smoke)),
                                 np.round(np.random.normal(123.1, 17.4, n_non))))
births.show(5)

---

<a id='2'></a>
## 2. Step 1 - explore the two groups

Before testing anything, look at the groups. How many in each, and how do they
differ?


In [ ]:
births.group('Maternal Smoker')

In [ ]:
means_table = births.group('Maternal Smoker', np.average)
means_table

> **Note the column name.** `group(col, np.average)` renames the aggregated
> column to `Birth Weight average`. Using `np.mean` instead gives
> `Birth Weight mean`. Check which you have before calling `.column()` on it.


In [ ]:
# Overlaid histograms show the two distributions on the same axis
births.hist('Birth Weight', group='Maternal Smoker')
plots.title('Birth weight by maternal smoking')
plots.show()

The two shapes overlap heavily, but the smokers' distribution sits to the left.
The question is whether a gap that size could appear by chance.


---

<a id='3'></a>
## 3. Step 2 - the observed difference

The test statistic is the difference between the two group averages. **Decide the
direction and keep it consistent** - every simulated value must be computed the
same way as the observed one.


In [ ]:
observed_difference = (means_table.column(1).item(1)
                       - means_table.column(1).item(0))
observed_difference

`.item(1)` is the True group (smokers), `.item(0)` is False - `group` sorts the
labels, and `False` comes before `True`.

So a negative value means smokers' babies weigh less.


---

<a id='4'></a>
## 4. Step 3 - simulating under the null by shuffling

Here is the whole idea.

**If the null is true, the labels are meaningless.** A baby's weight has nothing
to do with which group it was put in. So you could shuffle the labels, reassign
them at random, and the difference in group averages should look much like the
one you actually saw.

Do that a few thousand times and you learn what differences chance produces.


In [ ]:
# Shuffle just the label column. sample(with_replacement=False) on a
# one-column table returns those values in a random order.
shuffled_labels = births.select('Maternal Smoker').sample(with_replacement=False)
shuffled_labels.show(5)

In [ ]:
# Attach the shuffled labels to the ORIGINAL weights
simulated_births = Table().with_columns(
    'Shuffled Maternal Smoker', shuffled_labels.column(0),
    'Birth Weight',             births.column('Birth Weight'))
simulated_births.show(5)

In [ ]:
# The difference under one shuffle -- usually small
means_array = simulated_births.group('Shuffled Maternal Smoker', np.average).column(1)
means_array.item(1) - means_array.item(0)

Run that cell a few times. The value moves around zero, because the labels no
longer mean anything.

Now wrap it in a function so it can be repeated.


In [ ]:
def simulated_difference(table, group_label, numeric_label, function):
    """One trial: shuffle the labels, then recompute the difference in statistics."""
    shuffled_group = table.select(group_label).sample(with_replacement=False)
    simulated_table = Table().with_columns(
        group_label,   shuffled_group.column(0),
        numeric_label, table.column(numeric_label))
    stats_array = simulated_table.group(group_label, function).column(1)
    return stats_array.item(1) - stats_array.item(0)

simulated_difference(births, 'Maternal Smoker', 'Birth Weight', np.average)

In [ ]:
simulated_differences = make_array()

for i in np.arange(2500):
    one_difference = simulated_difference(births, 'Maternal Smoker',
                                          'Birth Weight', np.average)
    simulated_differences = np.append(simulated_differences, one_difference)

len(simulated_differences)

---

<a id='5'></a>
## 5. Step 4 - the p-value

Plot what chance produces, and mark where the observed value falls.


In [ ]:
Table().with_column('Difference Between Group Means',
                    simulated_differences).hist()
plots.scatter(observed_difference, -0.002, color='red', s=60, zorder=3)
plots.title('Simulated differences if the null were true')
plots.show()

print('observed:', round(observed_difference, 2))

The red dot is nowhere near the simulated values. The **p-value** puts a number
on that: the proportion of simulations at least as extreme as what you saw.


In [ ]:
empirical_p = np.average(simulated_differences <= observed_difference)
empirical_p

Zero here means it did not happen once in 2,500 trials - not that it is
impossible. With more simulations you would get a small number rather than zero.

By convention, a p-value below 5% is called *statistically significant*, and
below 1% *highly* so. Those are conventions, not laws of nature - the cutoff is
a choice you make before looking.

> **What a small p-value means.** The data would be surprising if the null were
> true. That is all. It does not tell you the null is false, how big the effect
> is, or what caused it.


---

<a id='6'></a>
## 6. One-sided or two-sided?

The direction of your alternative decides the statistic.

| Alternative | Statistic | p-value |
|---|---|---|
| Group B is **greater** | `mean_B - mean_A` | `np.average(simulated >= observed)` |
| Group B is **smaller** | `mean_B - mean_A` | `np.average(simulated <= observed)` |
| They are **different**, either way | `abs(mean_B - mean_A)` | `np.average(simulated >= observed)` |

Take the absolute value only when you genuinely do not care which direction, and
say so before you look at the data. Choosing afterwards is how you find effects
that are not there.


In [ ]:
def simulated_abs_difference(table, group_label, numeric_label, function):
    """Two-sided version: the size of the difference, ignoring direction."""
    shuffled_group = table.select(group_label).sample(with_replacement=False)
    simulated_table = Table().with_columns(
        group_label,   shuffled_group.column(0),
        numeric_label, table.column(numeric_label))
    stats_array = simulated_table.group(group_label, function).column(1)
    return abs(stats_array.item(1) - stats_array.item(0))

simulated_abs_difference(births, 'Maternal Smoker', 'Birth Weight', np.average)

---

<a id='7'></a>
## 7. Why shuffling is not enough: confounding

The test above says the difference is unlikely to be chance. It does **not** say
smoking caused it.

Nobody assigned these mothers to smoke. They chose, and the groups may differ in
other ways - age, income, diet, access to care. Any of those could produce the
same gap.

A variable that differs between the groups and also affects the outcome is a
**confounder**. It is why an observational study can establish association and
never causation, no matter how small the p-value.


In [ ]:
# A confounder in miniature: two treatments with identical recovery rates
# within each severity group -- but different rates overall.
patients = Table().with_columns(
    'Severity',  make_array('mild', 'mild', 'severe', 'severe'),
    'Treatment', make_array('A', 'B', 'A', 'B'),
    'Patients',  make_array(20, 180, 180, 20),
    'Recovered', make_array(18, 162, 90, 10))
patients = patients.with_column(
    'Rate', np.round(patients.column('Recovered') / patients.column('Patients'), 2))
patients

In [ ]:
# Within each severity level, A and B do equally well.
# Overall, B looks far better -- because B was mostly given to mild cases.
for t in ['A', 'B']:
    rows = patients.where('Treatment', t)
    overall = sum(rows.column('Recovered')) / sum(rows.column('Patients'))
    print(f'Treatment {t}: overall recovery {overall:.0%}')

Severity is the confounder. Compare the treatments without accounting for it and
you conclude B is better; it isn't.

This is not a subtle statistical error - it is what happens whenever the groups
were not formed at random.


---

<a id='8'></a>
## 8. Randomised controlled trials

There is one way to rule out confounders you have not thought of: **assign the
groups yourself, at random.**

Then no variable - measured or not - can systematically differ between the
groups except by chance. And chance is exactly what the A/B test accounts for.

That is why an RCT supports a causal conclusion and an observational study does
not, even with identical data and an identical p-value.

The example below is a real trial: botulinum toxin A for chronic back pain,
patients randomly assigned to treatment or placebo.


In [ ]:
# A stand-in for bta.csv: 31 patients, 1 = pain relief, 0 = none
back_pain = Table().with_columns(
    'Group',  np.append(np.repeat('Control', 16), np.repeat('Treatment', 15)),
    'Result', np.append(np.append(np.repeat(1, 2), np.repeat(0, 14)),
                        np.append(np.repeat(1, 9), np.repeat(0, 6))))
back_pain.pivot('Result', 'Group')

In [ ]:
means_array = back_pain.group('Group', np.average).column('Result average')
observed_difference = abs(means_array.item(1) - means_array.item(0))
observed_difference

In [ ]:
simulated_differences = make_array()

for i in np.arange(2500):
    one_difference = simulated_abs_difference(back_pain, 'Group', 'Result', np.average)
    simulated_differences = np.append(simulated_differences, one_difference)

Table().with_column('Simulated Differences Between Group Averages',
                    simulated_differences).hist()
plots.scatter(observed_difference, -0.02, color='red', s=60, zorder=3)
plots.title('If the treatment made no difference')
plots.show()

In [ ]:
np.average(simulated_differences >= observed_difference)

Same machinery as the birth-weight test - shuffle, recompute, compare. The
**arithmetic is identical.**

What differs is the design. Because patients were randomly assigned, there is no
confounder to worry about, and the conclusion can be causal: the treatment
relieved pain.

That distinction lives in how the data was collected, not in anything the code
can see. No amount of analysis recovers it after the fact.


---

<a id='9'></a>
## 9. Quick reference

### The four steps

| Step | Code |
|---|---|
| 1 Explore | `tbl.group(label, np.average)` · `tbl.hist(col, group=label)` |
| 2 Observe | `means.column(1).item(1) - means.column(1).item(0)` |
| 3 Simulate | shuffle labels, recompute, repeat |
| 4 Conclude | `np.average(simulated >= observed)` |

### The shuffle, in three lines

```python
shuffled = tbl.select(label).sample(with_replacement=False)
sim = Table().with_columns(label, shuffled.column(0),
                           value, tbl.column(value))
stats = sim.group(label, np.average).column(1)
```

### Direction

| Alternative | Statistic | Comparison |
|---|---|---|
| B greater | `mean_B - mean_A` | `>= observed` |
| B smaller | `mean_B - mean_A` | `<= observed` |
| Different either way | `abs(mean_B - mean_A)` | `>= observed` |

### Study design

| | Groups formed by | Supports |
|---|---|---|
| Observational | the subjects themselves | association only |
| Randomised controlled trial | the researcher, at random | causation |

### Things that catch people out

| | |
|---|---|
| `group(col, np.average)` | renames the column to `... average` |
| `group` row order | alphabetical, so `False` is `.item(0)` |
| Shuffling with replacement | wrong - it changes the group sizes |
| Choosing one-sided after looking | finds effects that are not there |
| A small p-value | says the data is surprising, not that the null is false |
| A significant result from observational data | still cannot establish causation |

---

### Where this comes from in the textbook

- [Chapter 12 - Comparing Two Samples](https://inferentialthinking.com/chapters/12/Comparing_Two_Samples.html)
- [Chapter 12.1 - A/B testing](https://inferentialthinking.com/chapters/12/1/AB_Testing.html)
- [Chapter 12.2 - Deflategate](https://inferentialthinking.com/chapters/12/2/Deflategate.html)
- [Chapter 12.3 - Causality](https://inferentialthinking.com/chapters/12/3/Causality.html)
